<a href="https://githubtocolab.com/PauloSBLima/pipeline-sanitizacao/blob/main/sanitizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 👓 Sobre

In [1]:
#=========================================================
# MINI-PROJETO: Pipeline de ETL com Python Puro
#=========================================================
# Aqui você encontrará a estrutura de como ler arquivos CSV 
# e realizar a higienização dos dados utilizando apenas 
# Python puro, sem bibliotecas externas.

## 📕 1.0 Importação das Bibliotecas

In [2]:
import csv                    
import urllib.request                       # Módulo nativo para fazer downloads da internet
import re                                   # Módulo nativo para expressões regulares - REGEX
from datetime import datetime               # Módulo nativo para manipulação de datas e horas
import minhas_funcoes as mf                 # Módulo criado por mim para organizar as funções auxiliares
from typing import List, Dict, Any, Union   # Dicas de tipo (type hints). Elas servem para documentar o código.

print(f"{mf.NEGRITO}{mf.VERDE}*** ✓ 1.0 Bibliotecas importadas com sucesso ***{mf.RESET}")

*** ✓ 1.0 Bibliotecas importadas com sucesso ***


## 📂 2.0 Execução - Arquivos de Dados

### 2.1 Dados de Produtos (olist_products_dataset.csv)

In [3]:
# Variáveis para o Relatório de Status Manual
total_linhas = 0
total_cancelados = 0
total_nulos_corrigidos = 0

# Criar a lista de dicionário para armazenar os dados limpos:
produtos_sanitizados = []

print("Iniciando a leitura dos dados - Produtos...")

# Arquivo contendo os Produtos - Link do arquivo CSV "Raw" no GitHub:
url_csv_raw_products = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_products_dataset.csv"

try:
    # 1. Fazendo a requisição na internet
    resposta = urllib.request.urlopen(url_csv_raw_products)
    
    # 2. Decodificando a resposta corretamente linha por linha
    linhas_texto = [linha.decode('utf-8') for list_linha in [resposta.readlines()] for linha in list_linha]

    # 3. Mapeando o CSV para um leitor de dicionários
    # Pega aquela lista de strings do passo anterior e a transforma em um leitor que mapeia cada linha do arquivo como se fosse um dicionário,
    # onde as chaves são os nomes das colunas do CSV 
    leitor_online = csv.DictReader(linhas_texto)
    cabecalho = leitor_online.fieldnames
    # Preparação para o cabeçalho do arquivo sanitizado:  
    cabecalho_saida = cabecalho
    print(f"Campos do CSV de Produtos: {cabecalho}")

    # Tradução dos campos do CSV de Produtos:
    # 01. product_id                     : ID do produto
    # 02. product_category_name          : Nome da categoria do produto
    # 03. product_name_lenght            : Comprimento do nome do produto
    # 04. product_description_lenght     : Comprimento da descrição do produto
    # 05. product_photos_qty             : Quantidade de fotos do produto
    # 06. product_weight_g               : Peso do produto em gramas
    # 07. product_length_cm              : Comprimento do produto em centímetros
    # 08. product_height_cm              : Altura do produto em centímetros
    # 09. product_width_cm               : Largura do produto em centímetros
    # A T E N Ç Ã O: Os campos 03 e 04 apresentam um erro de digitação, onde "length" foi escrito como "lenght".

    # 4. Conversão imediata para lista de dicionários (Evita o erro de String)
    produtos = list(leitor_online)
    qtd_prods = len(produtos)
    print(f"\nTotal de produtos originais: {qtd_prods:,}".replace(",", "."))
    print("-" * 100)

    # 5. Prévia dos dados de entrada
    mf.ExibirDados("Alguns dados de entrada:", produtos, 5)

    # 6. Verificação de Nulos na base de dados
    mf.ContarValoresNulos(produtos, "product_id", "Total de produtos com o id do produto nulo")
    mf.ContarValoresNulos(produtos, "product_category_name", "Total de produtos com o nome da categoria nulo")
    mf.ContarValoresNulos(produtos, "product_name_lenght", "Total de produtos com o tamanho do nome nulo")
    mf.ContarValoresNulos(produtos, "product_description_lenght", "Total de produtos com a descrição nula")
    mf.ContarValoresNulos(produtos, "product_photos_qty", "Total de produtos com a quantidade de fotos nula")
    mf.ContarValoresNulos(produtos, "product_weight_g", "Total de produtos com peso nulo")
    mf.ContarValoresNulos(produtos, "product_length_cm", "Total de produtos com comprimento nulo")
    mf.ContarValoresNulos(produtos, "product_height_cm", "Total de produtos com altura nula")
    mf.ContarValoresNulos(produtos, "product_width_cm", "Total de produtos com largura nula")
    print("-" * 100)

    """
     Executando as linhas acima verificamos:
           32.951 registros de produtos
              610 produtos com o nome da categoria nulo, que representa cerca de 1,85% do total de produtos da base, o que é um número relativamente 
                  baixo considerando o total de produtos (32.951). O impacto no viés do modelo seria praticamente nulo, e evitaríamos de "poluir" os dados com suposições,
                  se eliminássemos os registros nulos. Contudo, foi solicitado para corrigir os valores nulos, preenchendo-os com "sem categoria".
              610 produtos com o tamanho do nome nulos, mesmo percentual de 1,85%. Se o percentual fosse maior, e considerando que o campo é numérico,
                  uma solução seria preencher os valores nulos com a média ou mediana do campo, e criar uma coluna indicadora de nulos para esse campo
                  (tipo "product_name_lenght_was_null" onde o valor é 1 ou 0, indicando se o valor original era nulo ou não). Mas, como já iremos 
                  inserir os "sem categoria", então a probabilidade destes registros serems os mesmos do caso acima é muito alta, vamos optar por preencher os valores nulos
                  do tamanho do nome do produto com a mediana, e criar a coluna indicadora de nulos para esse campo, para evitar de "poluir" os dados com suposições.
              610 produtos com as descrições nulas. Exatamente o mesmo raciocínio do tópico anterior.
              610 produtos com a quantidade de fotos nula. Idem do caso acima.
                2 produtos com peso nulo, que representa cerca de 0,006% do total de produtos da base, o que é um número extremamente baixo considerando o total de
                  produtos (32.951). Então nestes casos vamos optar por não considerar os registros (linhas inteiras),
                2 produtos com comprimento nulo. Idem do caso acima.  
                2 produtos com altura nula. Idem do caso acima.
                2 produtos com largura nula. Idem do caso acima.

     Importante (Gatilho Mental):

      Imputar a Mediana (ou Média):
           Substituir os valores nulos pela mediana dos valores válidos é a abordagem ideal para a maioria dos modelos
           (como Regressão Linear, Redes Neurais e Modelos Baseados em Árvores).
           Colocar a mediana no lugar do nulo coloca o registro no "centro da massa" dos dados.
           Isso minimiza o impacto negativo em algoritmos que calculam distâncias ou derivadas 
           pois o valor imputado não gera um erro absurdo.

      Por que a Mediana e não a Média?
           O tamanho do nome de um produto (product_name_length) costuma ser um número inteiro pequeno 
           (ex: entre 10 e 70 caracteres). Se houver alguns produtos com nomes gigantescos (outliers),
            a média será puxada para cima. A mediana ignora esses extremos e pega o comportamento do "produto típico".

      Imputar 0 (Zero) — Cuidado com o viés:
           Colocar 0 pode parecer intuitivo (já que o nome não existe, o tamanho seria zero), mas essa escolha traz riscos dependendo do modelo.
           O problema do Zero: Para modelos lineares ou baseados em distância (como KNN, SVM ou Regressão Linear), o 0 será interpretado 
           como um valor real extremamente baixo, e não como "dado faltante". 
           O modelo vai achar que o produto tem um nome de comprimento zero, o que pode distorcer a linha de tendência (reta de regressão).
           Quando o Zero funciona? Se você for usar Modelos Baseados em Árvores (Random Forest, XGBoost, LightGBM), 
           o 0 funciona bem. As árvores de decisão conseguem isolar o 0 em um "ramo" específico facilmente 
           (ex: se tamanho <= 0 vá para a esquerda). Porém, se já criou a coluna booleana, o 0 se torna redundante para a árvore.
    """

    # 7. Cálculo prévio das medianas necessárias para imputação
    # Preferi utilizar a mediana pois a média é facilmente "enganada" por comportamentos extremos, que são os "Outliers".
    # A média é um indicador extremamente sensível. Ela soma todos os valores e divide pelo total, o que significa que um único 
    # número absurdamente alto pode puxar a média para cima, distorcendo a realidade.
    mediana_nome = mf.CalcularMedianaColuna(produtos, "product_name_lenght")
    print(f"\nMediana do comprimento do nome dos produtos: {mediana_nome}")

    mediana_descricao = mf.CalcularMedianaColuna(produtos, "product_description_lenght")
    print(f"Mediana do comprimento da descrição dos produtos: {mediana_descricao}")

    mediana_fotos = mf.CalcularMedianaColuna(produtos, "product_photos_qty")
    print(f"Mediana da quantidade de fotos dos produtos: {mediana_fotos}")

    # 8. Loop de sanitização linha por linha
    for linha in produtos:
        total_linhas += 1
        nulo_corrigido = 0 
        produto = {} 
        
        # [01] ID do produto
        produto['product_id'] = linha['product_id']

        # [06, 07] CRITÉRIO DE EXCLUSÃO: Peso ou Comprimento nulos causam o descarte do registro
        if linha['product_weight_g'] in mf.TERMOS_NULOS or linha['product_length_cm'] in mf.TERMOS_NULOS:
            total_cancelados += 1
            continue  # Pula para o próximo produto sem salvar este

        # [02] product_category_name
        if linha['product_category_name'] in mf.TERMOS_NULOS:
            produto['product_category_name'] = "sem categoria"
            nulo_corrigido = 1
        else:
            produto['product_category_name'] = mf.NormalizarString(linha['product_category_name'], "sem categoria", "minusculas")

        # [03] product_name_lenght (Mediana + Flag Booleana)
        if linha['product_name_lenght'] in mf.TERMOS_NULOS:
            # A coluna "product_name" não veio no CSV, então não temos como recuperar o nome do produto para calcular o comprimento do nome.
            # Os valores estão vazios creio que por falha de não ter vindo a coluna "product_name", então se preencher com 0 ou com "sem nome" ou algo do tipo, isso pode prejudicar a análise dos dados.
            # O ideal seria incluir "sem_nome" para seguir o mesmo padrão utilizado para os demais conteúdos deste campo, mas como é um campo numérico, por esse motivo,
            # optamos por incluir o valor da mediana e criar a coluna "product_name_lenght_was_null" (booleana-1/0) para facilitar futuras análises.
            produto['product_name_lenght'] = mediana_nome
            produto['product_name_lenght_was_null'] = 1
            nulo_corrigido = 1
        else:
            produto['product_name_lenght'] = mf.NormalizarValorInteiro(linha['product_name_lenght'])
            produto['product_name_lenght_was_null'] = 0

        # [04] product_description_lenght
        if linha['product_description_lenght'] in mf.TERMOS_NULOS:
            produto['product_description_lenght'] = mediana_descricao
            produto['product_description_lenght_was_null'] = 1
            nulo_corrigido = 1
        else:
            produto['product_description_lenght'] = mf.NormalizarValorInteiro(linha['product_description_lenght'])
            produto['product_description_lenght_was_null'] = 0

        # [05] product_photos_qty
        if linha['product_photos_qty'] in mf.TERMOS_NULOS:
            produto['product_photos_qty'] = mediana_fotos
            produto['product_photos_qty_was_null'] = 1
            nulo_corrigido = 1
        else:
            produto['product_photos_qty'] = mf.NormalizarValorInteiro(linha['product_photos_qty'])
            produto['product_photos_qty_was_null'] = 0
      
        # [06, 07, 08, 09] Demais dados (Já validados contra nulos acima)
        produto['product_weight_g'] = mf.NormalizarValorInteiro(linha['product_weight_g'])
        produto['product_length_cm'] = mf.NormalizarValorInteiro(linha['product_length_cm'])
        produto['product_height_cm'] = mf.NormalizarValorInteiro(linha['product_height_cm'])
        produto['product_width_cm'] = mf.NormalizarValorInteiro(linha['product_width_cm'])

        # Contabiliza se esta linha teve alguma correção
        if nulo_corrigido == 1:
            total_nulos_corrigidos += 1

        # Adiciona o produto limpo na lista final
        produtos_sanitizados.append(produto)      

except Exception as e:
    print(f"\n{mf.NEGRITO}{mf.VERMELHO}❌Erro ao tentar acessar a URL: {e}{mf.RESET}")
    print("Dica: Verifique se a estrutura dos dados recebidos mudou ou se há oscilação de rede.")

Iniciando a leitura dos dados - Produtos...
Campos do CSV de Produtos: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Total de produtos originais: 32.951
----------------------------------------------------------------------------------------------------
Alguns dados de entrada:
['1e9e8ef04dbcff4541ed26657ea517e5', 'perfumaria', '40', '287', '1', '225', '16', '10', '14']
['3aa071139cb16b67ca9e5dea641aaa2f', 'artes', '44', '276', '1', '1000', '30', '18', '20']
['96bd76ec8810374ed1b65e291975717f', 'esporte_lazer', '46', '250', '1', '154', '18', '9', '15']
['cef67bcfe19066a932b7673e239eb23d', 'bebes', '27', '261', '1', '371', '26', '4', '26']
['9dc1a7de274444849c219cff195d0b71', 'utilidades_domesticas', '37', '402', '4', '625', '20', '17', '13']
------------------------------------------------------------------------------------------------

## 📄 Relatório de Status da base SANITIZADA:

In [4]:
print("\nSUMÁRIO ESTATÍSTICO\n")

print("Base de PRODUTOS:")
print(f"\n{mf.NEGRITO}{mf.AMARELO}✓ Leitura e higienização dos dados concluídas.{mf.RESET}")
print(f"Total de linhas processadas: {total_linhas:,}".replace(",", "."))
print(f"Total de registros descartados (Peso/Dimensão ausentes): {total_cancelados:,}".replace(",", "."))
print(f"Total de registros que continham nulos e foram corrigidos: {total_nulos_corrigidos:,}".replace(",", "."))
print(f"Total de registros na nova base limpa: {mf.NEGRITO}{mf.AMARELO}{len(produtos_sanitizados):,}{mf.RESET}".replace(",", "."))


SUMÁRIO ESTATÍSTICO

Base de PRODUTOS:

✓ Leitura e higienização dos dados concluídas.
Total de linhas processadas: 32.951
Total de registros descartados (Peso/Dimensão ausentes): 2
Total de registros que continham nulos e foram corrigidos: 609
Total de registros na nova base limpa: 32.949


No resultado anterior, o total de registros que continham nulos e foram corrigidos,<br>
imaginava-se que daria os 610, porém um dos registros cancelados também seria limpo.<br>
Coincidência matemática.

## 🔍Conferência visual

In [5]:
# Conferência visual para a geração do novo csv de saída
# Lista os 20 primeiros produtos, transforma cada um em texto e quebra a linha
print("\n".join(str(prod) for prod in produtos_sanitizados[:20]))

print(f"\nTotal de produtos sanitizados: {mf.NEGRITO}{mf.VERDE}{len(produtos_sanitizados):,}{mf.RESET}".replace(",", "."))

{'product_id': '1e9e8ef04dbcff4541ed26657ea517e5', 'product_category_name': 'perfumaria', 'product_name_lenght': 40, 'product_name_lenght_was_null': 0, 'product_description_lenght': 287, 'product_description_lenght_was_null': 0, 'product_photos_qty': 1, 'product_photos_qty_was_null': 0, 'product_weight_g': 225, 'product_length_cm': 16, 'product_height_cm': 10, 'product_width_cm': 14}
{'product_id': '3aa071139cb16b67ca9e5dea641aaa2f', 'product_category_name': 'artes', 'product_name_lenght': 44, 'product_name_lenght_was_null': 0, 'product_description_lenght': 276, 'product_description_lenght_was_null': 0, 'product_photos_qty': 1, 'product_photos_qty_was_null': 0, 'product_weight_g': 1000, 'product_length_cm': 30, 'product_height_cm': 18, 'product_width_cm': 20}
{'product_id': '96bd76ec8810374ed1b65e291975717f', 'product_category_name': 'esporte_lazer', 'product_name_lenght': 46, 'product_name_lenght_was_null': 0, 'product_description_lenght': 250, 'product_description_lenght_was_null': 0

## 💾 Geração do novo csv sanitizado de PRODUTOS:

In [6]:
mf.GerarArquivoCSV(produtos_sanitizados, "olist_products_dataset_sanitizado.csv")

Novo cabeçalho gerado com 12 colunas: ['product_id', 'product_category_name', 'product_name_lenght', 'product_name_lenght_was_null', 'product_description_lenght', 'product_description_lenght_was_null', 'product_photos_qty', 'product_photos_qty_was_null', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Iniciando a gravação do arquivo: olist_products_dataset_sanitizado.csv...
✓ Sucesso! Arquivo gerado com 32.949 linhas.


# =============================================
### 2.2 Dados de Pedidos de Compra (olist_orders_dataset.csv)

In [7]:
# Criar a lista de dicionário para armazenar os dados limpos:
pedidos_sanitizados = []

print("Iniciando a leitura dos dados - Pedidos...")

# Arquivo contendo os Produtos - Link do arquivo CSV "Raw" no GitHub:
url_csv_raw_orders = "https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_orders_dataset.csv"

try:
    # 1. Fazendo a requisição na internet
    resposta = urllib.request.urlopen(url_csv_raw_orders)
    
    # 2. Decodificando a resposta corretamente linha por linha
    linhas_texto = [linha.decode('utf-8') for linha in resposta.readlines()]

    # 3. Mapeando o CSV para um leitor de dicionários
    leitor_online = csv.DictReader(linhas_texto)
    cabecalho = leitor_online.fieldnames
    # Preparação para o cabeçalho do arquivo sanitizado:  
    cabecalho_saida = cabecalho
    print(f"Campos do CSV de Pedidos de Compra: {cabecalho}")

    # Tradução dos campos do CSV de Pedidos de Compra:
    # 01. order_id                       : ID do pedido
    # 02. customer_id                    : ID do cliente
    # 03. order_status                   : Status do pedido
    # 04. order_purchase_timestamp       : Data e hora da compra
    # 05. order_approved_at              : Data e hora da aprovação do pedido
    # 06. order_delivered_carrier_date   : Data e hora da entrega para a transportadora
    # 07. order_delivered_customer_date  : Data e hora da entrega para o cliente
    # 08. order_estimated_delivery_date  : Data e hora da entrega estimada
    
    # 4. Conversão imediata para lista de dicionários (Evita o erro de String)
    pedidos = list(leitor_online)
    qtd_pedidos = len(pedidos)
    print(f"\nTotal de pedidos originais: {qtd_pedidos:,}".replace(",", "."))
    print("-" * 100)

    # 5. Prévia dos dados de entrada
    mf.ExibirDados("Alguns dados de entrada:", pedidos, 5)

    # 6. Verificação de Nulos na base de dados
    mf.ContarValoresNulos(pedidos, "order_id", "Total de pedidos com o id do pedido nulo")
    mf.ContarValoresNulos(pedidos, "customer_id", "Total de pedidos com o id do cliente nulo")
    mf.ContarValoresNulos(pedidos, "order_status", "Total de pedidos com status nulo")
    mf.ContarValoresNulos(pedidos, "order_purchase_timestamp", "Total de pedidos com data da compra nula")
    mf.ContarValoresNulos(pedidos, "order_approved_at", "Total de pedidos com data da aprovação do pedido nula")
    mf.ContarValoresNulos(pedidos, "order_delivered_carrier_date", "Total de pedidos com data de entrega para a transportadora nula")
    mf.ContarValoresNulos(pedidos, "order_delivered_customer_date", "Total de pedidos com data de entrega ao cliente nula")
    mf.ContarValoresNulos(pedidos, "order_estimated_delivery_date", "Total de pedidos com data de entrega estimada nula")
    print("-" * 100)

    # Executando as linhas acima verificamos:
    #       99.441 registros de pedidos de compra
    #          160 pedidos com data de aprovação do pedido nula. Faz sentido haver contéudo nulo se por exemplo, um pedido foi cancelado antes de ser aprovado.
    #        1.783 pedidos com data de entrega para a transportadora nula, que representa cerca de 1,8% do total de pedidos, o que é um número relativamente
    #              baixo considerando o total de pedidos (99.441), porém faz sentido haver contéudo nulo se por exemplo, um pedido foi cancelado antes de ser entregue.
    #        2.965 pedidos com data de entrega ao cliente nula, que representa cerca de 3% do total de pedidos, o que é um número relativamente 
    #              baixo, porém faz sentido haver contéudo nulo se por exemplo, um pedido foi cancelado.
    #
    #       Diante disso, não podemos simplesmente eliminar os registros que apresentem valores nulos. Uma solução é criar uma nova coluna para saber quais 
    #       contéudos eram nulos e usar a mediana para substituir os valores nulos.
    
    # 7. Cálculo prévio das medianas necessárias para imputação
    mediana_dt_aprovacao = mf.CalcularMedianaDataColuna(pedidos, "order_approved_at")
    print(f"\nMediana da data de aprovação do pedido: {mediana_dt_aprovacao}")

    mediana_dt_entrega_transportadora = mf.CalcularMedianaDataColuna(pedidos, "order_delivered_carrier_date")
    print(f"Mediana da data de entrega para a transportadora: {mediana_dt_entrega_transportadora}")
   
    mediana_dt_entrega_cliente = mf.CalcularMedianaDataColuna(pedidos, "order_delivered_customer_date")
    print(f"Mediana da data de entrega ao cliente: {mediana_dt_entrega_cliente}")
    
    # Reseta os totalizadores para o loop de sanitização
    total_cancelados = 0
    total_linhas = 0
    total_nulos_corrigidos = 0

    # 8. Criar um dicionário vazio para armazenar as contagens (Solicitação da diretoria)
    total_por_status_order = {}

    # 9. Loop de sanitização linha por linha
    for linha in pedidos:
        total_linhas += 1
        nulo_corrigido = 0 
        pedido = {} 
        
        # [01] ID do pedido
        pedido['order_id'] = linha['order_id']

        # [02] ID do cliente
        pedido['customer_id'] = linha['customer_id']

        # [03] Status do pedido
        pedido['order_status'] = linha['order_status']

        # [04] Data e hora da compra
        pedido['order_purchase_timestamp'] = mf.FormatarData(linha['order_purchase_timestamp'])

        # [05] Data e hora da aprovação do pedido
        if linha['order_approved_at'] in mf.TERMOS_NULOS:
            pedido['order_approved_at_wass_null'] = 1
            pedido['order_approved_at'] = mediana_dt_aprovacao
            nulo_corrigido = 1
        else:
            pedido['order_approved_at_wass_null'] = 0
            pedido['order_approved_at'] = mf.FormatarData(linha['order_approved_at'])
            
        # [06] Data e hora da entrega para a transportadora
        if linha['order_delivered_carrier_date'] in mf.TERMOS_NULOS:
            pedido['order_delivered_carrier_date_wass_null'] = 1
            pedido['order_delivered_carrier_date'] = mediana_dt_entrega_transportadora
            nulo_corrigido = 1
        else:
            pedido['order_delivered_carrier_date_wass_null'] = 0
            pedido['order_delivered_carrier_date'] = mf.FormatarData(linha['order_delivered_carrier_date'])

        # [07] Data e hora da entrega para o cliente
        if linha['order_delivered_customer_date'] in mf.TERMOS_NULOS:
            pedido['order_delivered_customer_date_wass_null'] = 1
            pedido['order_delivered_customer_date'] = mediana_dt_entrega_cliente
            nulo_corrigido = 1
            # Vamos verificar os status dos pedidos tendo em vista a solicitação da diretoria
            # Verificar se o alto volume de nulos na data de entrega ao cliente é devido OBRIGATORIAMENTE a pedidos cancelados.
            status_atual = linha['order_status']
            if status_atual in total_por_status_order:
                total_por_status_order[status_atual] += 1
            else:
                total_por_status_order[status_atual] = 1
        else:
            pedido['order_delivered_customer_date_wass_null'] = 0
            pedido['order_delivered_customer_date'] = mf.FormatarData(linha['order_delivered_customer_date'])

        # [08] Data e hora da entrega estimada
        pedido['order_estimated_delivery_date'] = mf.FormatarData(linha['order_estimated_delivery_date'])

        # Contabiliza se esta linha teve alguma correção
        if nulo_corrigido == 1:
            total_nulos_corrigidos += 1

        # Adiciona o pedido limpo na lista final
        pedidos_sanitizados.append(pedido)

except Exception as e:
    print(f"\n{mf.NEGRITO}{mf.VERMELHO}❌Erro ao tentar acessar a URL: {e}{mf.RESET}")
    print("Dica: Verifique se a estrutura dos dados recebidos mudou ou se há oscilação de rede.")

Iniciando a leitura dos dados - Pedidos...
Campos do CSV de Pedidos de Compra: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Total de pedidos originais: 99.441
----------------------------------------------------------------------------------------------------
Alguns dados de entrada:
['e481f51cbdc54678b7cc49136f2d6af7', '9ef432eb6251297304e76186b10a928d', 'delivered', '2017-10-02 10:56:33', '2017-10-02 11:07:15', '2017-10-04 19:55:00', '2017-10-10 21:25:13', '2017-10-18 00:00:00']
['53cdb2fc8bc7dce0b6741e2150273451', 'b0830fb4747a6c6d20dea0b8c802d7ef', 'delivered', '2018-07-24 20:41:37', '2018-07-26 03:24:27', '2018-07-26 14:31:00', '2018-08-07 15:27:45', '2018-08-13 00:00:00']
['47770eb9100c2d0c44946d9cf07ec65d', '41ce2a54c0b03bf3443c3d931a367089', 'delivered', '2018-08-08 08:38:49', '2018-08-08 08:55:23', '2018-08-08 13:50:00', '2018-08-17

## 📄 Relatório de Status da base SANITIZADA:

In [8]:
print("\nSUMÁRIO ESTATÍSTICO\n")

print("Base de PEDIDOS:")
print(f"\n{mf.NEGRITO}{mf.AMARELO}✓ Leitura e higienização dos dados concluídas.{mf.RESET}")
print(f"Total de linhas processadas: {total_linhas:,}".replace(",", "."))
print(f"Total de registros descartados: {total_cancelados:,}".replace(",", "."))
print(f"Total de registros que continham nulos e foram corrigidos: {total_nulos_corrigidos:,}".replace(",", "."))
print(f"Total de registros na nova base limpa: {mf.NEGRITO}{mf.AMARELO}{len(pedidos_sanitizados):,}{mf.RESET}".replace(",", "."))


SUMÁRIO ESTATÍSTICO

Base de PEDIDOS:

✓ Leitura e higienização dos dados concluídas.
Total de linhas processadas: 99.441
Total de registros descartados: 0
Total de registros que continham nulos e foram corrigidos: 2.980
Total de registros na nova base limpa: 99.441


## 🔍Conferência visual

In [9]:
# Conferência visual para a geração do novo csv de saída
# Lista os 20 primeiros pedidos, transforma cada um em texto e quebra a linha
print("\n".join(str(pedido) for pedido in pedidos_sanitizados[:20]))

print(f"\nTotal de pedidos sanitizados: {mf.NEGRITO}{mf.VERDE}{len(pedidos_sanitizados):,}{mf.RESET}".replace(",", "."))

{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '02/10/2017', 'order_approved_at_wass_null': 0, 'order_approved_at': '02/10/2017', 'order_delivered_carrier_date_wass_null': 0, 'order_delivered_carrier_date': '04/10/2017', 'order_delivered_customer_date_wass_null': 0, 'order_delivered_customer_date': '10/10/2017', 'order_estimated_delivery_date': '18/10/2017'}
{'order_id': '53cdb2fc8bc7dce0b6741e2150273451', 'customer_id': 'b0830fb4747a6c6d20dea0b8c802d7ef', 'order_status': 'delivered', 'order_purchase_timestamp': '24/07/2018', 'order_approved_at_wass_null': 0, 'order_approved_at': '26/07/2018', 'order_delivered_carrier_date_wass_null': 0, 'order_delivered_carrier_date': '26/07/2018', 'order_delivered_customer_date_wass_null': 0, 'order_delivered_customer_date': '07/08/2018', 'order_estimated_delivery_date': '13/08/2018'}
{'order_id': '47770eb9100c2d0c44946d9cf07ec65d', 'customer

## 💾 Geração do novo csv sanitizado de PEDIDOS:

In [10]:
mf.GerarArquivoCSV(pedidos_sanitizados, "olist_orders_dataset_sanitizado.csv")

Novo cabeçalho gerado com 11 colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at_wass_null', 'order_approved_at', 'order_delivered_carrier_date_wass_null', 'order_delivered_carrier_date', 'order_delivered_customer_date_wass_null', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Iniciando a gravação do arquivo: olist_orders_dataset_sanitizado.csv...
✓ Sucesso! Arquivo gerado com 99.441 linhas.


In [11]:
# Exibir os resultados da solicitação da diretoria
print("\nTotalização por Status (condição: Data de entrega ao cliente nula)\n")

# Antes de exibir os resultados, vamos ordenar o dicionário por ordem decrescente dos totalizadores
status_ordenados = sorted(total_por_status_order.items(), key=lambda item: item[1], reverse=True)

for status, total in status_ordenados:
    print(f"Status: {status} | Total: {total:,}".replace(",", "."))


Totalização por Status (condição: Data de entrega ao cliente nula)

Status: shipped | Total: 1.107
Status: canceled | Total: 619
Status: unavailable | Total: 609
Status: invoiced | Total: 314
Status: processing | Total: 301
Status: delivered | Total: 8
Status: created | Total: 5
Status: approved | Total: 2


## 📌 Solicitação de Análise pela Diretoria:
De acordo com o resultado acima, verifica-se que a suposição da diretoria não se concretizou.<br>
Nem todos os registros nulos da data de entrega ao cliente foram OBRIGATORIAMENTE cancelados.<br>
A maioria foi de embarcados (shipped=1.107) seguido aí sim, pelos cancelados (619) e os demais listados acima.

❌ Suspeita sobre as datas de entrega ao cliente
# =============================================